# Lab 58: Measuring lost-in-the-middle

Long-context models attend best to the start and end of the context and worst to the middle, so a perfect retriever can still be wrong if it places the gold passage in the middle. Build the harness that measures this position bias: sweep the gold position, recover the curve, and quantify mitigations. Concept: [concepts/rag/lost-in-the-middle.md](../../concepts/rag/lost-in-the-middle.md). Fill in the `TODO` cells.

## Step 0: Setup

In [ ]:
from lostmiddle import (position_sweep, recall_prob, answer_correct,
                        mean_accuracy_random_placement, rerank_to_top_accuracy)
# Liu et al. (2023): long-context models attend best to the start and end of the context, worst to
# the middle - accuracy by gold position is U-shaped. For RAG, a perfect retriever can still be
# wrong if it puts the answer in the middle. The answerer here is a deterministic stand-in for that
# positional bias; swap in real model calls over your corpus to measure your own stack.
print("measuring accuracy by gold position (the methodology, not a fixed result)")

## Step 1: Sweep the gold position and read the curve

In [ ]:
# TODO: call position_sweep(20) and print accuracy by position. Where is accuracy highest and
# lowest? Compare the edge positions (1 and 20) to the middle (10).
raise NotImplementedError

## Step 2: Mean accuracy hides the bias

In [ ]:
# Mean accuracy hides the bias. A retriever that places the gold passage at a random position
# reports a single number that sits well below what the edges achieve.
K = 20
curve = position_sweep(K)

mean = mean_accuracy_random_placement(K)
print(f"mean accuracy (random gold position): {mean:.2f}")
print(f"edge accuracy:                        {(curve[1]+curve[K])/2:.2f}")
print("Reporting only the mean would hide that the middle is failing. Report accuracy BY position.")

## Step 3: Mitigations - rerank-to-top and shrink-k

In [ ]:
# TODO: compare mean_accuracy_random_placement(20) to rerank_to_top_accuracy(20), and compare
# the middle-position accuracy at k=40 vs k=6. Why does shrinking the context help the middle?
raise NotImplementedError

## What you built

The measurement methodology for lost-in-the-middle. `lostmiddle.py` sweeps the gold passage across every position in the context and reports accuracy *by position*, recovering the U-shaped curve (edges ~0.95, middle ~0.50 here), shows that **mean accuracy hides the bias** (a random-placement retriever reports ~0.64, far below the edges), and quantifies two mitigations: reranking the gold passage toward the top recovers edge-level accuracy, and shrinking the context (k=40 → k=6) lifts middle accuracy from 0.50 to 0.77 because the dead middle is an *absolute* edge-distance effect - fewer passages means no deep middle to get lost in.

**Where this simplifies:** the answerer is a deterministic stand-in whose correctness follows a U-shaped positional bias - it is *not* a measurement of any real model. The deliverable is the harness: swap `answer_correct` for real model calls over your own corpus and gold set, and the same sweep measures your stack's actual position curve. The absolute-edge-window model (high within ~5 passages of either end) matches the qualitative finding; the real curve's shape, window, and depth are model- and prompt-specific, which is exactly why you measure rather than assume. The headline for RAG: retrieval recall is necessary but not sufficient - *where* the retriever places the evidence in the prompt is part of the eval.